In [ ]:
# log_reg_model

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# --- 데이터 로드 및 샘플링 ---
# 전체 데이터셋이 너무 크므로 50%만 무작위로 추출하여 사용합니다.
train = pd.read_csv("data/raw/train.csv")
train_sampled = train.sample(frac=0.5, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# --- 파이프라인 구성 ---
# n_features=2**18은 그대로 유지하되, 데이터 샘플링으로 메모리 문제를 해결합니다.
vec_hash_35 = HashingVectorizer(
    analyzer='char', 
    ngram_range=(3, 5), 
    n_features=2**18, 
    alternate_sign=False, 
    lowercase=False
)

# 하이퍼파라미터 최적화 적용
log_reg_model = LogisticRegression(
    solver='saga',      # 대규모 데이터셋에 효율적인 솔버
    random_state=42,    
    class_weight='balanced', # 불균형 데이터셋에 대한 가중치 부여
    n_jobs=-1,          # 모든 CPU 코어를 사용하여 병렬 처리
    penalty='l2',       # 기본값인 L2 규제 적용
    C=1.0,              # 규제 강도 (기본값)
    max_iter=100        # 최대 반복 횟수 (기본값)
)
pipe_log_reg_model = Pipeline([("vec", vec_hash_35), ("clf", log_reg_model)])

# --- 교차 검증 및 평가 ---
print("\n--- 로지스틱 회귀 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    pipe_log_reg_model.fit(X_train, y_train)
    
    val_preds_proba = pipe_log_reg_model.predict_proba(X_val)[:, 1]
    
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)
    
oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> 로지스틱 회귀 모델 평균 ROC-AUC: {oof_score:.4f}")

In [2]:
# log_reg_model

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# --- 데이터 로드 및 샘플링 ---
# 50%만 무작위로 추출하여 사용합니다.
train = pd.read_csv("../data/raw/train.csv")
train_sampled = train.sample(frac=0.5, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# --- 파이프라인 구성 ---
vec_hash_35 = HashingVectorizer(analyzer='char', ngram_range=(3, 5), n_features=2**18, alternate_sign=False, lowercase=False)

# 하이퍼파라미터 명시적 추가
log_reg_model = LogisticRegression(
    solver='liblinear', 
    random_state=42, 
    class_weight='balanced',
    penalty='l2',  # 기본값 명시
    C=1.0,         # 규제 강도 (예시)
    max_iter=100   # 최대 반복 횟수 (예시)
)
pipe_log_reg_model = Pipeline([("vec", vec_hash_35), ("clf", log_reg_model)])

# --- 교차 검증 및 평가 ---
print("\n--- 로지스틱 회귀 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    pipe_log_reg_model.fit(X_train, y_train)
    val_preds_proba = pipe_log_reg_model.predict_proba(X_val)[:, 1]
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)
    
oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> 로지스틱 회귀 모델 평균 ROC-AUC: {oof_score:.4f}")


--- 로지스틱 회귀 모델 교차 검증 시작 ---
  - Fold 1 / 5
  - Fold 2 / 5
  - Fold 3 / 5
  - Fold 4 / 5
  - Fold 5 / 5

-> 로지스틱 회귀 모델 평균 ROC-AUC: 0.9714


In [6]:
# svc_model

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# --- 데이터 로드 및 샘플링 ---
# 전체 데이터셋이 너무 크므로 50%만 무작위로 추출하여 사용합니다.
train = pd.read_csv("../data/raw/train.csv")
train_sampled = train.sample(frac=0.5, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# --- 파이프라인 구성 ---
vec_hash_35 = HashingVectorizer(analyzer='char', ngram_range=(3, 5), n_features=2**18, alternate_sign=False, lowercase=False)

# 하이퍼파라미터 최적화 적용 (n_jobs=-1 제거)
svc_model = CalibratedClassifierCV(
    LinearSVC(
        random_state=42, 
        class_weight='balanced', # 불균형 데이터셋에 대한 가중치 부여
    ), 
    cv=3 # 캘리브레이션 내부 교차 검증 폴드 수 (메모리 최적화)
)
pipe_svc_model = Pipeline([("vec", vec_hash_35), ("clf", svc_model)])

# --- 교차 검증 및 평가 ---
print("\n--- 선형 SVM 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    pipe_svc_model.fit(X_train, y_train)
    val_preds_proba = pipe_svc_model.predict_proba(X_val)[:, 1]
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)

oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> 선형 SVM 모델 평균 ROC-AUC: {oof_score:.4f}")


--- 선형 SVM 모델 교차 검증 시작 ---
  - Fold 1 / 5
  - Fold 2 / 5
  - Fold 3 / 5
  - Fold 4 / 5
  - Fold 5 / 5

-> 선형 SVM 모델 평균 ROC-AUC: 0.9720


In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# --- 데이터 로드 및 샘플링 ---
# 데이터 샘플링 비율을 20%로 낮춥니다.
train = pd.read_csv("../data/raw/train.csv")
train_sampled = train.sample(frac=0.2, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# --- 파이프라인 구성 ---
vec_tfidf_tree = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 4),
    sublinear_tf=True,
    lowercase=False,
    max_features=50000
)
svd_300 = TruncatedSVD(n_components=300, random_state=4321)

# 하이퍼파라미터 최적화 적용
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
    n_estimators=300,  # 트리 개수를 300개로 줄였습니다.
    max_depth=20
)
pipe_rf_model = Pipeline([("vec", vec_tfidf_tree), ("svd", svd_300), ("clf", rf_model)])

# --- 교차 검증 및 평가 ---
print("\n--- 랜덤 포레스트 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    pipe_rf_model.fit(X_train, y_train)
    val_preds_proba = pipe_rf_model.predict_proba(X_val)[:, 1]
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)

oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> 랜덤 포레스트 모델 평균 ROC-AUC: {oof_score:.4f}")


--- 랜덤 포레스트 모델 교차 검증 시작 ---
  - Fold 1 / 5
  - Fold 2 / 5
  - Fold 3 / 5
  - Fold 4 / 5
  - Fold 5 / 5

-> 랜덤 포레스트 모델 평균 ROC-AUC: 0.9387


In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# --- 데이터 로드 및 샘플링 ---
train = pd.read_csv("../data/raw/train.csv")
train_sampled = train.sample(frac=0.2, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# --- 파이프라인 구성 ---
vec_tfidf_tree = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 4),
    sublinear_tf=True,
    lowercase=False,
    max_features=50000
)

# 하이퍼파라미터 최적화 적용 (GPU 설정 제거)
lgbm_model = lgb.LGBMClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
pipe_lgbm_model = Pipeline([("vec", vec_tfidf_tree), ("clf", lgbm_model)])

# --- 교차 검증 및 평가 ---
print("\n--- LightGBM 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    pipe_lgbm_model.fit(X_train, y_train)
    val_preds_proba = pipe_lgbm_model.predict_proba(X_val)[:, 1]
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)

oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> LightGBM 모델 평균 ROC-AUC: {oof_score:.4f}")


--- LightGBM 모델 교차 검증 시작 ---
  - Fold 1 / 5
[LightGBM] [Info] Number of positive: 249851, number of negative: 869357
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 54.136439 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1683068
[LightGBM] [Info] Number of data points in the train set: 1119208, number of used features: 49969
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
  - Fold 2 / 5
[LightGBM] [Info] Number of positive: 249852, number of negative: 869357
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 57.231963 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1684953
[LightGBM] [Info] Number of data points in the train set: 1119209, number of used features: 49971
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> init

In [4]:
# xgb_model

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# --- 데이터 로드 및 샘플링 ---
# 메모리 문제 해결을 위해 20%만 무작위로 추출하여 사용합니다.
train = pd.read_csv("../data/raw/train.csv")
train_sampled = train.sample(frac=0.2, random_state=42)

X = train_sampled["URL"]
y = train_sampled["label"]

# 불균형 데이터 처리를 위한 'scale_pos_weight' 계산
positive_count = sum(y == 1)
negative_count = sum(y == 0)
scale_pos_weight_value = negative_count / positive_count

# --- 파이프라인 구성 ---
vec_tfidf_tree = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 4),
    sublinear_tf=True,
    lowercase=False,
    max_features=50000
)
svd_300 = TruncatedSVD(n_components=300, random_state=4321)

# 하이퍼파라미터 최적화 적용
xgb_model = xgb.XGBClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight_value, # 불균형 데이터셋에 대한 가중치 부여
    n_jobs=-1,                               # 모든 CPU 코어를 사용하여 병렬 처리
    eval_metric='logloss',                   # 경고 메시지 방지를 위해 eval_metric 명시
    use_label_encoder=False                  # 경고 메시지 방지
)
pipe_xgb_model = Pipeline([("vec", vec_tfidf_tree), ("svd", svd_300), ("clf", xgb_model)])

# --- 교차 검증 및 평가 ---
print("\n--- XGBoost 모델 교차 검증 시작 ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_proba, oof_labels = [], []

for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"  - Fold {i+1} / 5")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    pipe_xgb_model.fit(X_train, y_train)
    val_preds_proba = pipe_xgb_model.predict_proba(X_val)[:, 1]
    oof_preds_proba.extend(val_preds_proba)
    oof_labels.extend(y_val)

oof_score = roc_auc_score(oof_labels, oof_preds_proba)
print(f"\n-> XGBoost 모델 평균 ROC-AUC: {oof_score:.4f}")


--- XGBoost 모델 교차 검증 시작 ---
  - Fold 1 / 5


/usr/local/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [11:51:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  - Fold 2 / 5


/usr/local/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [11:55:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  - Fold 3 / 5


...

  - Fold 4 / 5


/usr/local/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:03:12] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  - Fold 5 / 5


/usr/local/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:08:30] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



-> XGBoost 모델 평균 ROC-AUC: 0.9393
